# 3-2. Advanced RAG: Reranking & Context Compression

## 학습 목표

- 검색된 문서를 그대로 사용하지 않고 재정렬하거나 압축한다.
- 답변에 필요한 핵심 근거만 정리한다.


## 공통 전제

- 실습 데이터: `../data/공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md`
- 기본 모델: `gpt-4o-mini`
- 기본 임베딩 모델: `text-embedding-3-small`
- 벡터DB: Qdrant Docker Server (`http://localhost:6333`)

> 이 노트북은 `2-2_vector_embedding_store_qdrant_md.ipynb`에서 저장한 Qdrant collection을 재사용한다.


In [1]:
from dotenv import load_dotenv

load_dotenv(override=True, dotenv_path="../../.env")

False

## 1. 검색 결과 가져오기


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from const.const import create_model, create_embedding

from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "civil_complaint_manual_medium"

EMBEDDING_MODEL = "text-embedding-3-small"

embeddings = create_embedding()

vector_store = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    url=QDRANT_URL,
)

/Users/parkchanryong/Desktop/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


In [3]:
question = "권장시간이 지났는데도 계속 상담을 요구하면 어떻게 하나요?"

docs = vector_store.similarity_search(question, k=6)

for i, doc in enumerate(docs, start=1):
    print(f"--- 검색 결과 {i} ---")
    print(doc.metadata)
    print(doc.page_content[:400].replace("\n", " "))
    print()

--- 검색 결과 1 ---
{'source': '공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md', 'page': 7, 'section': '일반적인 민원 응대요령', 'topic': '장시간 통화', 'document_type': 'preprocessed_markdown', 'chunk_type': 'page', 'chunk_id': 8, 'chunk_size': 700, 'chunk_overlap': 120, 'chunk_strategy': 'medium', 'embedding_model': 'text-embedding-3-small', 'collection_name': 'civil_complaint_manual_medium', '_id': '2e320b5e-5475-465d-b2be-b8a0b7eacad9', '_collection_name': 'civil_complaint_manual_medium'}
2-2 정당한 사유 없는 장시간 통화 ※ 권장시간 20분 설정의 경우 핵심대응 용건위주 질문유도 및 장시간 전화상담 곤란 안내 2-4 폭언(욕설, 협박, 성희롱 등) 핵심대응 전화 종료 및 증거확보를 통한 후속조치(법적조치 등)로 경각심 부여 ‣ ~~민원전화 수신 시부터 녹음하여 위법행위 증거 확보~~ ‣ ~~15분 이상 통화 장시간 상담 곤란 안내 및 용건 위주 질문유도~~ ※ 민원 신청은 문서가 원칙임을 안내, 지속시 신문고 등 접수 유도 ‣ ~~20분 이상 통화 통화 지속 곤란 안내 후 종료~~ 선생님, 죄송하지만 상담 권장시간이 초과되어 다른 민원 처리를 위해서(또는 대기하고 계시는 다른 민원인이 계셔서) 통화를 종료하오니 양해하여 주시기 바랍니다. ‣ ~~부서장 보고, 휴게시간 부여 등~~ 장시간 

--- 검색 결과 2 ---
{'source': '공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md', 'page': 7, 'section': '일반적인 민원 응대요령', 'topic': '장시간 통화', 'document

## 2. 간단한 규칙 기반 Reranking

질문 안의 핵심 키워드가 문서에 많이 포함될수록 높은 점수를 준다.


In [4]:
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


def simple_rerank(question: str, docs: list[Document]) -> list[Document]:
    keywords = ["권장시간", "장시간", "통화", "상담", "종료", "서면", "온라인"]

    scored = []

    for doc in docs:
        score = 0
        for keyword in keywords:
            if keyword in doc.page_content:
                score += 1
        scored.append((score, doc))

    # 점수 높은 순으로 정렬
    scored.sort(key=lambda x: x[0], reverse=True)

    return [doc for score, doc in scored]


reranked_docs = simple_rerank(question, docs)

for i, doc in enumerate(reranked_docs, start=1):
    print(f"--- 재정렬 결과 {i} ---")
    print(doc.metadata)
    print(doc.page_content[:400].replace("\n", " "))
    print()

--- 재정렬 결과 1 ---
{'source': '공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md', 'page': 7, 'section': '일반적인 민원 응대요령', 'topic': '장시간 통화', 'document_type': 'preprocessed_markdown', 'chunk_type': 'page', 'chunk_id': 8, 'chunk_size': 700, 'chunk_overlap': 120, 'chunk_strategy': 'medium', 'embedding_model': 'text-embedding-3-small', 'collection_name': 'civil_complaint_manual_medium', '_id': '2e320b5e-5475-465d-b2be-b8a0b7eacad9', '_collection_name': 'civil_complaint_manual_medium'}
2-2 정당한 사유 없는 장시간 통화 ※ 권장시간 20분 설정의 경우 핵심대응 용건위주 질문유도 및 장시간 전화상담 곤란 안내 2-4 폭언(욕설, 협박, 성희롱 등) 핵심대응 전화 종료 및 증거확보를 통한 후속조치(법적조치 등)로 경각심 부여 ‣ ~~민원전화 수신 시부터 녹음하여 위법행위 증거 확보~~ ‣ ~~15분 이상 통화 장시간 상담 곤란 안내 및 용건 위주 질문유도~~ ※ 민원 신청은 문서가 원칙임을 안내, 지속시 신문고 등 접수 유도 ‣ ~~20분 이상 통화 통화 지속 곤란 안내 후 종료~~ 선생님, 죄송하지만 상담 권장시간이 초과되어 다른 민원 처리를 위해서(또는 대기하고 계시는 다른 민원인이 계셔서) 통화를 종료하오니 양해하여 주시기 바랍니다. ‣ ~~부서장 보고, 휴게시간 부여 등~~ 장시간 

--- 재정렬 결과 2 ---
{'source': '공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md', 'page': 7, 'section': '일반적인 민원 응대요령', 'topic': '장시간 통화', 'docume

## 3. Context Compression

검색 문단 전체를 LLM에 넣지 않고, 답변에 필요한 핵심 근거만 요약한다.


In [ ]:
def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(
        [
            f"[근거 {i}] page={doc.metadata.get('page')}, chunk={doc.metadata.get('chunk_id')}\n{doc.page_content}"
            for i, doc in enumerate(docs, start=1)
        ]
    )


# 상위 4개만 Context Compression에 사용
context = format_docs(reranked_docs[:4])

compression_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
검색된 문단에서 질문 답변에 필요한 핵심 근거만 정리하세요.
문단에 없는 내용은 추가하지 마세요.
""",
        ),
        (
            "human",
            """
질문:
{question}

검색 문단:
{context}

핵심 근거를 항목별로 정리하세요.
""",
        ),
    ]
)

llm = create_model()

compression_chain = compression_prompt | llm | StrOutputParser()

compressed_context = compression_chain.invoke(
    {"question": question, "context": context}
)

print(compressed_context)

/Users/parkchanryong/Desktop/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found z-ai/glm-5.3-flash in available_models, but type is unknown and inference may fail.
  warnings.warn(


# 권장시간 초과 후 지속 상담 요구 시 대응 (근거 1: 2-2 정당한 사유 없는 장시간 통화)

## 1. 핵심대응 원칙
- **용건위주 질문유도 및 장시간 전화상담 곤란 안내**

## 2. 단계별 대응
| 구분 | 대응 방법 |
|------|----------|
| 15분 이상 통화 | 장시간 상담 곤란 안내 및 용건 위주 질문유도 |
| 20분 이상 통화 | 통화 지속 곤란 안내 후 **종료** |

## 3. 종료 안내 멘트(예시)
> "선생님, 죄송하지만 상담 권장시간이 초과되어 다른 민원 처리를 위해서(또는 대기하고 계시는 다른 민원인이 계셔서) 통화를 종료하오니 양해하여 주시기 바랍니다."

## 4. 사후 조치
- **부서장 보고** 및 휴게시간 부여
- 장시간 통화를 한 부서원에게 **30분 이내 휴게시간 부여 가능**
- 통화 내용과 정도를 감안하여 필요시 **위법행위 관리대장 작성하여 채증**


## 4. 압축된 근거로 답변 생성


In [ ]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 공직자 민원응대 매뉴얼 기반 업무지원 AI입니다.
            제공된 근거를 바탕으로만 답변하세요. 질문에 대한 근거가 없으면 없다고 대답하세요.

            답변 형식:
            1. 핵심 대응
            2. 단계별 조치
            3. 안내 표현
            4. 주의사항
            """,
        ),
        (
            "human",
            """
질문:
{question}

핵심 근거:
{context}
""",
        ),
    ]
)

answer_chain = answer_prompt | llm | StrOutputParser()

answer = answer_chain.invoke({"question": question, "context": compressed_context})

print(answer)

## 1. 핵심 대응

권장시간 초과 후에도 상담을 계속 요구하는 경우, **용건 위주 질문을 유도**하고 **장시간 전화상담이 곤란함을 안내**하는 것이 원칙입니다.

## 2. 단계별 조치

| 구분 | 대응 방법 |
|------|----------|
| **15분 이상 통화 시** | 장시간 상담 곤란을 안내하고, 용건 위주로 질문 유도 |
| **20분 이상 통화 시** | 통화 지속이 곤란함을 안내한 후 **통화 종료** |

## 3. 안내 표현

> "선생님, 죄송하지만 상담 권장시간이 초과되어 다른 민원 처리를 위해서(또는 대기하고 계시는 다른 민원인이 계셔서) 통화를 종료하오니 양해하여 주시기 바랍니다."

## 4. 주의사항

- 통화 종료 후 반드시 **부서장에게 보고**해야 합니다.
- 장시간 통화를 한 부서원에게는 **30분 이내 휴게시간 부여가 가능**합니다.
- 통화 내용과 정도를 감안하여 필요시 **위법행위 관리대장을 작성하여 채증**하십시오.


## 핵심 정리

- Reranking은 검색 결과의 순서를 다시 조정하는 과정이다.
- Context Compression은 답변에 필요한 근거만 줄여서 LLM에 전달하는 과정이다.
- Modular RAG에서는 `retrieve`, `rerank`, `compress`, `generate`를 각각 분리할 수 있다.
